In [15]:
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


In [2]:
import polars as pl
print(pl.__version__)


1.34.0


In [3]:
import os
# limit threads to avoid memory spikes
os.environ["POLARS_MAX_THREADS"] = "4"

In [27]:
import polars as pl
import gc

df = pl.scan_parquet("processed_github_features.parquet")

BASE_COLS = [
    "repo_name",
    "day",
    "total_stars_scaled",
    # add ONLY non-star predictors you want
    # "total_forks_scaled",
    # "total_commits_scaled",
    # "total_issues_opened_scaled",
    # "total_prs_merged_scaled",
]


In [28]:
df = df.sort(["repo_name", "day"])

In [29]:
df = df.with_columns(
    pl.col("total_stars_scaled")
      .shift(-7)
      .over("repo_name")
      .alias("target_stars_7d")
)

# materialize a manageable subset EARLY
df_small = (
    df
    .select(BASE_COLS + ["target_stars_7d"])
    .drop_nulls("target_stars_7d")
    .head(500_000)      # choose size you can afford
    .collect()
)

del df
gc.collect()


93

In [30]:
df_small = df_small.sort(["repo_name", "day"])

df_small = df_small.with_row_index("idx")

n = df_small.height
cutoff = int(0.8 * n)

train_df = df_small.filter(pl.col("idx") < cutoff).drop("idx")
test_df  = df_small.filter(pl.col("idx") >= cutoff).drop("idx")

del df_small
gc.collect()


0

In [31]:
DROP_COLS = ["repo_name", "day", "target_stars_7d"]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train = train_df.select(feature_cols)
y_train = train_df["target_stars_7d"]

X_test  = test_df.select(feature_cols)
y_test  = test_df["target_stars_7d"]

del train_df, test_df
gc.collect()


0

In [21]:

lin_reg_model = LinearRegression()
ridge_reg_model = Ridge(alpha=1.0)
rand_f_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE
    )
grad_boost_model =  GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

In [32]:
lin_reg_model.fit(X_train, y_train)
ridge_reg_model.fit(X_train, y_train)
rand_f_model.fit(X_train, y_train)
grad_boost_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [33]:
y_pred_xgb   = xgb_model.predict(X_test)
y_pred_gb    = grad_boost_model.predict(X_test)
y_pred_rf    = rand_f_model.predict(X_test)
y_pred_ridge = ridge_reg_model.predict(X_test)
y_pred_lin   = lin_reg_model.predict(X_test)

In [34]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

In [35]:
results_lin = evaluate(y_test, y_pred_lin),
results_ridge = evaluate(y_test, y_pred_ridge),
results_rfr = evaluate(y_test, y_pred_rf),
results_grad_boost =  evaluate(y_test, y_pred_gb),
results_xgb_boost =  evaluate(y_test, y_pred_xgb),


In [36]:
print("Liniar: ",results_lin)
print("Ridge: ",results_ridge)
print("RandomForest: ",results_rfr)
print("GradientBoost: ",results_grad_boost)
print("XGBboost: ",results_xgb_boost)


Liniar:  ({'MAE': 0.018470535054802895, 'RMSE': np.float64(0.029371376478321172), 'R2': 0.9761179089546204},)
Ridge:  ({'MAE': 0.018472121655920923, 'RMSE': np.float64(0.029371285450601456), 'R2': 0.9761180664046462},)
RandomForest:  ({'MAE': 0.017948469326534735, 'RMSE': np.float64(0.02909308880261708), 'R2': 0.9765683299398843},)
GradientBoost:  ({'MAE': 0.018021992681962085, 'RMSE': np.float64(0.029097969722692456), 'R2': 0.9765604670619832},)
XGBboost:  ({'MAE': 0.018096037209033966, 'RMSE': np.float64(0.029104219225770264), 'R2': 0.9765504002571106},)


Liniar:  ({'MAE': 0.01847054436802864, 'RMSE': np.float64(0.02936997631606821), 'R2': 0.9761201739311218},)

Ridge:  ({'MAE': 0.018471021935158295, 'RMSE': np.float64(0.029369485981123163), 'R2': 0.9761209926295951},)

RandomForest:  ({'MAE': 0.01801262627248385, 'RMSE': np.float64(0.029084251767435183), 'R2': 0.9765825625327753},)

GradientBoost:  ({'MAE': 0.01801112247178043, 'RMSE': np.float64(0.029045039193027335), 'R2': 0.9766456646541579},)

XGBboost:  ({'MAE': 0.027615739032626152, 'RMSE': np.float64(0.03638374502026424), 'R2': 0.9633530378341675},)

Liniar:  ({'MAE': 0.017610879614949226, 'RMSE': np.float64(0.027848251535198938), 'R2': 0.9785306453704834},)

Ridge:  ({'MAE': 0.0176108497597264, 'RMSE': np.float64(0.02784753918350099), 'R2': 0.9785317212483075},)

RandomForest:  ({'MAE': 0.014805813270959124, 'RMSE': np.float64(0.026631601811240656), 'R2': 0.980365576428189},)

GradientBoost:  ({'MAE': 0.014764464488608393, 'RMSE': np.float64(0.02648436532463581), 'R2': 0.9805820795273282},)

XGBboost:  ({'MAE': 0.01717722788453102, 'RMSE': np.float64(0.02760284048300847), 'R2': 0.9789073467254639},)

 bogdan_ichim@yahoo.com